# Train the Transformer (DonderFormer)

An encoder-decoder Transformer that writes a chart as event tokens (see `src/donderformer/models/donderformer.py` and `data/tokens.py`):

- **Encoder** reads a window of `past + target + lookahead` mel frames (default 512 + 256 + 64 = 832 frames, 19.1 s).
- **Decoder** sees `BOS <course> <level> <past events> SEP` and generates the target region's events as `TIME_k NOTE` pairs, where `TIME_k` is frame `k` of the window (23 ms grid).
- **Generation** slides the window by `target` frames; each window's past region is primed with the notes generated so far.

The output is the shared note format (one class id per 23 ms frame), so the evaluation below is the same as `03_train_lstm`'s autoregressive evaluation and `ChartExporter` can export both models side by side.

Set `TOY = True` for a quick end-to-end check on the toy dataset (tiny model, a few epochs, CPU is fine).

In [ ]:
import math
import time

import numpy as np
import torch
from torch.utils.data import DataLoader

from donderformer.data.chart_dataset import Augment, ChartWindowDataset, WindowConfig, collate
from donderformer.data.feature_extractor import FeatureExtractor
from donderformer.models.donderformer import DonderFormer, DonderFormerConfig, generate_chart
from donderformer.paths import DataPaths, outputs_dir
from donderformer.runs import new_run_dir, save_run
from donderformer.training import evaluate_tokens, train_epoch_tokens

TOY = False                                   # True: toy dataset, tiny model, short run

if TOY:
    paths = DataPaths.toy()
    SPLIT = "toy_all"                         # every course of the 12 toy songs
    PRESET = "tiny"
    EPOCHS, BATCH_SIZE, WARMUP_STEPS = 5, 16, 20       # a pipeline check, not a real run
    EVAL_DECODE_EVERY = 5
else:
    paths = DataPaths()                       # DONDER_DATA env var, else <repo>/data
    SPLIT = "oni_v1"                          # oni_v2 / all_v1 once 01_make_split has made them
    PRESET = "base"                           # base (~11.5M params), small (~4.3M), tiny (~1M)
    EPOCHS, BATCH_SIZE, WARMUP_STEPS = 200, 64, 1000
    EVAL_DECODE_EVERY = 5

extractor = FeatureExtractor(frame_ms=23)     # cache made by 02_extract_features
WINDOW = WindowConfig(past_frames=512, target_frames=256, lookahead_frames=64, max_tokens=640)
MODEL_CFG = DonderFormerConfig.preset(PRESET)
AUGMENT = Augment()                           # SpecAugment, level dropout, prefix corruption

LR = 3e-4
MIN_LR = 1e-5
WEIGHT_DECAY = 0.01
LABEL_SMOOTHING = 0.02
PATIENCE = 15                                 # epochs without a validation-loss gain before stopping
SAMPLING = dict(temperature=1.0, top_p=0.9)   # for test generation; validation decoding is greedy

MODEL_NAME = "donderformer"
ALIAS = PRESET + ("_toy" if TOY else "")
RUN_DIR = new_run_dir(MODEL_NAME, SPLIT, extractor.id, alias=ALIAS)
print("Run folder:", RUN_DIR)
HPARAMS = dict(preset=PRESET, epochs=EPOCHS, batch_size=BATCH_SIZE, warmup_steps=WARMUP_STEPS, lr=LR,
               min_lr=MIN_LR, weight_decay=WEIGHT_DECAY, label_smoothing=LABEL_SMOOTHING, patience=PATIENCE,
               sampling=SAMPLING, augment=AUGMENT.__dict__)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
# bf16 autocast on CUDA (A100/H100/H200); full precision elsewhere.
AMP_DTYPE = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else None
print("Device:", device, "| autocast:", AMP_DTYPE)
if device.type == "cuda":
    print(torch.cuda.get_device_name(device))

## Data

Training windows start at random frames and are redrawn every epoch (`resample`); validation and test windows tile each chart with hop `target_frames`. All mel of a part is held in memory (float16, about 1.4 GB for the full Oni train part).

In [ ]:
train_dataset = ChartWindowDataset(paths, SPLIT, extractor, "train", WINDOW, augment=AUGMENT, seed=0)
val_dataset = ChartWindowDataset(paths, SPLIT, extractor, "validation", WINDOW)
test_dataset = ChartWindowDataset(paths, SPLIT, extractor, "test", WINDOW)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, collate_fn=collate)
for name, ds in (("train", train_dataset), ("validation", val_dataset), ("test", test_dataset)):
    print(f"{name}: {len(ds.chart_ids)} charts, {len(ds)} windows")

# One training example, decoded
mel, tokens, sep = val_dataset[min(2, len(val_dataset) - 1)]
print("mel", tuple(mel.shape), "| tokens", len(tokens), "| SEP at", sep)
print(val_dataset.vocab.decode(tokens[:40]), "...")

## Model

In [ ]:
model = DonderFormer(MODEL_CFG, WINDOW).to(device)
print(f"{PRESET}: {model.n_parameters() / 1e6:.2f}M parameters, vocab {model.vocab.size}")

# Sanity check: shapes, and the loss starts near log(vocab)
mel, tokens, sep = collate([train_dataset[i] for i in range(4)])
with torch.no_grad():
    logits = model(mel.to(device), tokens.to(device))
print("logits", tuple(logits.shape), "| initial loss",
      round(evaluate_tokens(model, [(mel, tokens, sep)], device), 3), "vs log(vocab)", round(math.log(model.vocab.size), 3))

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.98), weight_decay=WEIGHT_DECAY)
TOTAL_STEPS = EPOCHS * len(train_loader)

def lr_at(step):
    """Linear warmup, then cosine down to MIN_LR."""
    if step < WARMUP_STEPS:
        return (step + 1) / WARMUP_STEPS
    progress = min(1.0, (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS))
    return (MIN_LR + (LR - MIN_LR) * 0.5 * (1 + math.cos(math.pi * progress))) / LR

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_at)

## Evaluation helpers

The scoring functions are copied from `03_train_lstm` unchanged, so both models are scored the same way: onset and onset+type F1 within ±1 frame, plus TaikoNation's DCHuman, OCHuman and pattern-space metrics.

In [ ]:
import json
import csv

def onset_metrics(true_seq, pred_seq, tolerance=1):
    true_onsets = np.where(true_seq != 0)[0]
    pred_onsets = np.where(pred_seq != 0)[0]

    matched_true = set()
    matches = 0

    for pred in pred_onsets:
        candidates = [
            t for t in true_onsets
            if abs(t - pred) <= tolerance and t not in matched_true
        ]

        if candidates:
            best = min(candidates, key=lambda t: abs(t - pred))
            matched_true.add(best)
            matches += 1

    precision = matches / len(pred_onsets) if len(pred_onsets) else 0
    recall = matches / len(true_onsets) if len(true_onsets) else 0

    if precision + recall:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = 0

    return precision, recall, f1, matches


def joint_onset_type_metrics(true_seq, pred_seq, tolerance=1):
    true_onsets = np.where(true_seq != 0)[0]
    pred_onsets = np.where(pred_seq != 0)[0]

    matched_true = set()
    matches = 0

    for pred in pred_onsets:
        candidates = [
            t for t in true_onsets
            if abs(t - pred) <= tolerance
            and true_seq[t] == pred_seq[pred]
            and t not in matched_true
        ]

        if candidates:
            best = min(candidates, key=lambda t: abs(t - pred))
            matched_true.add(best)
            matches += 1

    precision = matches / len(pred_onsets) if len(pred_onsets) else 0
    recall = matches / len(true_onsets) if len(true_onsets) else 0

    if precision + recall:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = 0

    return precision, recall, f1, matches


def remove_adjacent_notes(sequence):
    cleaned = sequence.copy()

    for t in range(1, len(sequence)):
        if sequence[t - 1] != 0 and sequence[t] != 0:
            cleaned[t] = 0

    return cleaned


def to_binary_chart(sequence):
    sequence = np.asarray(sequence)

    # Discrete notes
    binary = np.isin(sequence, [1, 2, 3, 4])

    # Drumroll / Denden: only count onset
    held = np.isin(sequence, [5, 6])
    starts = np.r_[True, sequence[1:] != sequence[:-1]]

    binary |= held & starts
    return binary.astype(np.uint8)


def paper_ochuman(human, ai):
    """
    Paper-style approximation:
    Exact binary agreement, with ±1 timestep
    tolerance for human note onsets.
    """
    human = np.asarray(human, dtype=bool)
    ai = np.asarray(ai, dtype=bool)

    assert len(human) == len(ai)

    correct = (human == ai)

    nearby = ai.copy()
    nearby[1:] |= ai[:-1]
    nearby[:-1] |= ai[1:]

    correct |= human & nearby

    return float(correct.mean())


def extract_patterns(binary_chart, window=8):
    chart = np.asarray(binary_chart, dtype=np.uint8)

    return {
        tuple(chart[i:i + window])
        for i in range(len(chart) - window + 1)
    }


def calculate_paper_metrics(human, generated):
    h = to_binary_chart(human)
    a = to_binary_chart(generated)

    dc = float(np.mean(h == a) * 100)
    oc = paper_ochuman(h, a) * 100

    hp = extract_patterns(h)
    ap = extract_patterns(a)

    over = len(ap) / 256 * 100
    hi = len(hp & ap) / len(hp) * 100 if hp else 0.0

    return {
        "DCHuman": dc,
        "OCHuman": oc,
        "Over_PSpace": over,
        "HI_PSpace": hi,
        "human_onsets": int(h.sum()),
        "ai_onsets": int(a.sum())
    }


def calculate_f1_metrics(human, predicted):
    p, r, f1, onset_matches = onset_metrics(
        human, predicted, tolerance=1
    )

    jp, jr, jf1, joint_matches = joint_onset_type_metrics(
        human, predicted, tolerance=1
    )

    return {
        "onset_precision": float(p),
        "onset_recall": float(r),
        "onset_f1": float(f1),
        "joint_precision": float(jp),
        "joint_recall": float(jr),
        "joint_f1": float(jf1),
        "onset_matches": int(onset_matches),
        "joint_matches": int(joint_matches)
    }


def human_sequence(dataset, chart_index):
    """The human chart in the shared format (one class per frame)."""
    from donderformer.data.tokens import events_to_classes
    return events_to_classes(dataset.events[chart_index], dataset.lengths[chart_index])


def generate_for(dataset, chart_index, **sampling):
    row = dataset.charts[chart_index]
    mel = dataset.mel[row["audio_id"]].astype(np.float32)
    return generate_chart(model, mel, row["course"], dataset.levels[chart_index], **sampling)


def decode_f1(dataset, n_charts=None, **sampling):
    """Mean onset and onset+type F1 of generated charts (greedy unless sampling is given)."""
    sampling = sampling or dict(temperature=0.0)
    scores = []
    for c in range(len(dataset.charts) if n_charts is None else min(n_charts, len(dataset.charts))):
        human, generated = human_sequence(dataset, c), generate_for(dataset, c, **sampling)
        scores.append((onset_metrics(human, generated)[2], joint_onset_type_metrics(human, generated)[2]))
    return tuple(float(x) for x in np.mean(scores, axis=0))

## Train

Validation loss is on target-region tokens only. Every `EVAL_DECODE_EVERY` epochs a few validation charts are generated greedily and scored. The best state by validation loss is kept; training stops after `PATIENCE` epochs without improvement.

In [ ]:
import copy

history = {"train": [], "val": [], "val_decode": []}
best = {"val": float("inf"), "epoch": -1, "state": None}
start_time = time.time()

for epoch in range(EPOCHS):
    train_dataset.resample(epoch)
    train_loss = train_epoch_tokens(model, train_loader, optimizer, device, scheduler,
                                    label_smoothing=LABEL_SMOOTHING, amp_dtype=AMP_DTYPE)
    val_loss = evaluate_tokens(model, val_loader, device, amp_dtype=AMP_DTYPE)
    history["train"].append(train_loss)
    history["val"].append(val_loss)
    line = (f"Epoch {epoch + 1:03d} | train {train_loss:.4f} | val {val_loss:.4f} | "
            f"lr {scheduler.get_last_lr()[0]:.2e} | {(time.time() - start_time) / 60:.1f} min")

    if (epoch + 1) % EVAL_DECODE_EVERY == 0:
        onset_f1, joint_f1 = decode_f1(val_dataset, n_charts=8)
        history["val_decode"].append({"epoch": epoch + 1, "onset_f1": onset_f1, "joint_f1": joint_f1})
        line += f" | val greedy onset F1 {onset_f1:.3f}, joint {joint_f1:.3f}"
    print(line, flush=True)

    if val_loss < best["val"]:
        best = {"val": val_loss, "epoch": epoch + 1, "state": copy.deepcopy(model.state_dict())}
    elif epoch + 1 - best["epoch"] >= PATIENCE:
        print(f"No validation gain for {PATIENCE} epochs; stopping")
        break

model.load_state_dict(best["state"])
print(f"Best epoch {best['epoch']}, val loss {best['val']:.4f}")

In [ ]:
checkpoint = save_run(
    RUN_DIR,
    model,
    config={"model": MODEL_NAME, "alias": ALIAS, "data": {"split": SPLIT, "features": extractor.id},
            "window": WINDOW.as_dict(), "model_config": MODEL_CFG.as_dict(), "hparams": HPARAMS,
            "n_parameters": model.n_parameters(), "best_epoch": best["epoch"], "device": str(device)},
    losses=history,
)
print("Saved", checkpoint)

# To evaluate an earlier run instead: run the cells up to "Model", then
# from donderformer.runs import load_run
# RUN_DIR = outputs_dir() / "runs" / MODEL_NAME / "<run folder name>"
# load_run(RUN_DIR, model, map_location=device)

## Autoregressive test evaluation

Same layout as `03_train_lstm`: one `<chart_id>.npz` per test chart in `eval/ar_test/charts/` (arrays `human`, `generated`, `clean`), per-chart metrics in `results.json` and means in `summary.csv`. Charts already generated are reused.

In [ ]:
EVAL_DIR = RUN_DIR / "eval"
SAVE_DIR = EVAL_DIR / "ar_test"
CHARTS_DIR = SAVE_DIR / "charts"
PLOTS_DIR = EVAL_DIR / "plots"
for d in (CHARTS_DIR, PLOTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

all_results = []
start_time = time.time()
for index, song_id in enumerate(test_dataset.chart_ids):
    save_path = CHARTS_DIR / f"{song_id}.npz"
    if save_path.exists():
        with np.load(save_path) as data:
            true_seq, generated, generated_clean = data["human"], data["generated"], data["clean"]
    else:
        true_seq = human_sequence(test_dataset, index)
        generated = generate_for(test_dataset, index, seed=42 + index + 1, **SAMPLING)
        assert len(generated) == len(true_seq)
        generated_clean = remove_adjacent_notes(generated)
        np.savez_compressed(save_path, human=true_seq, generated=generated, clean=generated_clean)

    result = {"song_id": str(song_id)}
    for label, predicted in [("raw", generated), ("clean", generated_clean)]:
        result[label] = {**calculate_f1_metrics(true_seq, predicted), **calculate_paper_metrics(true_seq, predicted)}
    all_results.append(result)
    with open(SAVE_DIR / "results.json", "w") as f:
        json.dump(all_results, f, indent=2)
    if (index + 1) % 10 == 0 or index + 1 == len(test_dataset.chart_ids):
        print(f"Evaluated {index + 1}/{len(test_dataset.chart_ids)} | {(time.time() - start_time) / 60:.1f} min", flush=True)

metrics = ["onset_precision", "onset_recall", "onset_f1", "joint_precision", "joint_recall", "joint_f1",
           "DCHuman", "OCHuman", "Over_PSpace", "HI_PSpace"]
summary = []
for label in ["raw", "clean"]:
    print(f"\n========== {label.upper()} ==========")
    for metric in metrics:
        mean = float(np.mean([r[label][metric] for r in all_results]))
        print(f"{metric:20s}: {mean:.4f}")
        summary.append({"version": label, "metric": metric, "mean": mean})
with open(SAVE_DIR / "summary.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["version", "metric", "mean"])
    writer.writeheader()
    writer.writerows(summary)
print("Metrics saved in", SAVE_DIR)

In [ ]:
import matplotlib.pyplot as plt

from donderformer.data.tja import CLASS_NAMES

song_id = test_dataset.chart_ids[0]
with np.load(CHARTS_DIR / f"{song_id}.npz") as data:
    human, generated = data["human"], data["generated"]

end = min(600, len(human))
colors = {1: "red", 2: "blue", 3: "darkred", 4: "darkblue", 5: "orange", 6: "purple"}
fig, axes = plt.subplots(2, 1, figsize=(16, 5), sharex=True)
for ax, seq, title in zip(axes, [human, generated], ["Human", "DonderFormer (autoregressive)"]):
    for note_type, color in colors.items():
        positions = np.where(seq[:end] == note_type)[0]
        ax.scatter(positions, np.ones(len(positions)), color=color, s=40, label=CLASS_NAMES[note_type])
    ax.set_title(f"{title}: {song_id}")
    ax.set_yticks([])
    ax.grid(axis="x", alpha=0.25)
axes[0].legend(loc="upper center", bbox_to_anchor=(0.5, 1.5), ncol=6)
axes[-1].set_xlabel("Frame (23 ms)")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "human_vs_autoregressive.png", dpi=200, bbox_inches="tight")
plt.show()

## Export to TJA

Writes playable charts next to the song audio (see `ChartExporter`). Add an LSTM run's `eval/ar_test/charts` folder to `sources` to export both models side by side.

In [ ]:
from donderformer.exporter import ChartExporter

exporter = ChartExporter(
    alias=f"donderformer_{ALIAS}",
    sources={"transformer": CHARTS_DIR},
    paths=paths,
    split=SPLIT,
    version="raw",
    include_human=True,
    n_songs=5,
)
exporter.export()